# Two-colour correction of the chromatic shear (error budget)

Two 48x48 pyramids on **either side** of the science wavelength: **rMod 3 at 400-500 nm** and **unmodulated at 700-800 nm**, fused per mode, science at **600 nm**, zenith 30 deg, seeing 0.9" along the line of sight. Flat photon spectrum (the 900 nm photons per nm at every wavelength). Reference: the best single stage sensing at 900 nm (700-1100 nm camera band, binning + rate by contrast, e2e SR-10 % rule).

Weights: `min_noise` = inverse variance (today's Kalman filter); `per_mode` = per-mode optimum of noise + aliasing + chromatic shear (the noise-free limit is the interpolation 0.30 / 0.70, sum w_i dn_i = 0). Chromatic shear: exact multi-beam form (`aoeb/twocolour.py`). Products: 450 / 750 nm, s0.9, 1300 corrected, staging root (WFSW untouched). Contrast at 600 nm, then scaled to 500-700 nm (same correction, contrast at each wavelength).

In [ ]:
%load_ext autoreload
%autoreload 2
import os, sys, warnings, importlib.util
os.environ['E2E_CALIB_DIR'] = '/raid1/mmenessini/calibration/WFSW_stage_twocolour'   # 450 / 750 products + links to the 900 ones
import numpy as np
import pandas as pd
import yaml
import matplotlib.pyplot as plt

E2E_DIR = os.path.abspath('aoeb/e2e')                      # run from SPECULA/config
sys.path.insert(0, E2E_DIR)
_spec = importlib.util.spec_from_file_location('aoeb_e2e_utils', os.path.join(E2E_DIR, 'utils.py'))
e2e = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(e2e)
from aoeb.chromatic import ChromaticModel
from aoeb.psf import PsfEngine
from aoeb.multiwfs import Study
from aoeb.twocolour import FlatPhotometry, TwoColourStudy
warnings.filterwarnings('ignore', message='.*(not calibrated at|not simulated for).*')

Y = yaml.safe_load(open('aoeb/e2e/ristretto_main.yml'))
m = e2e.machine()
assert m.paths.root == os.environ['E2E_CALIB_DIR'], m.paths.root
m.max_sensitivity = None
SEEING, ZENITH, SCI_NM = 0.9, Y['main']['zenithAngleInDeg'], 600.0
CONTROLLER = 'leaky'                                        # 'wiener': ~10-18 s per budget (background run)
LATENCY = Y['kpred']['delay'] * Y['main']['time_step']
AREA = (m.pupil() > 0).sum() * Y['main']['pixel_pitch'] ** 2
st = Study(m, PsfEngine(m), SEEING, LATENCY, CONTROLLER, AREA, bandw=Y['cred_vis']['bandw'], colour=0.0,
           cache_dir=f'results/twocolour_cache_{CONTROLLER}', zone=(2, 5), criterion='contrast')
cm = ChromaticModel(Y['atmo']['heights'], Y['atmo']['Cn2'], science_nm=SCI_NM, zenith_deg=ZENITH)
tc = TwoColourStudy(st, cm)
ph = FlatPhotometry(AREA, cm)
MAGS = (7.0, 8.0, 9.0, 10.0, 11.0, 12.0)                    # magnitude at 900 nm
BINS_ALL = st.binnings
dn = {k: (b['n_minus_1'] - tc.n_sci) / 1e-6 for k, b in (('400-500', ph.band(7, 400, 500)), ('700-800', ph.band(7, 700, 800)))}
dn['900'] = (float(cm.n_minus_1(900.)) - tc.n_sci) / 1e-6
print('dn vs 600 nm [1e-6]:', {k: round(v, 3) for k, v in dn.items()}, ' interpolation weight of 400-500:', round(-dn['700-800'] / (dn['400-500'] - dn['700-800']), 3))

In [ ]:
def ref900(rmod, mg, lo=700., hi=1100.):
    return [dict(rMod=rmod, n_subap=48, wavelength_nm=900., flux=ph.band(mg, lo, hi)['flux'])]

blue = lambda mg: ph.sensor(3.0, 48, mg, 400., 500.)
red = lambda mg: ph.sensor(0.0, 48, mg, 700., 800.)
CONFIGS = {   # name: (sensors(mag), weights, binnings, chromatic)
    'single 900 rMod 3': (lambda mg: ref900(3.0, mg), 'min_noise', BINS_ALL, True),
    'single 900 unmod': (lambda mg: ref900(0.0, mg), 'min_noise', BINS_ALL, True),
    'single 750 unmod': (lambda mg: [red(mg)], 'min_noise', (48,), True),
    'single 450 rMod 3': (lambda mg: [blue(mg)], 'min_noise', (48,), True),
    'fusion min-noise': (lambda mg: [blue(mg), red(mg)], 'min_noise', (48,), True),
    'fusion per-mode': (lambda mg: [blue(mg), red(mg)], 'per_mode', (48,), True),
    'fusion achromatic': (lambda mg: [blue(mg), red(mg)], 'min_noise', (48,), False),
    'single 900 rMod 3 achromatic': (lambda mg: ref900(3.0, mg), 'min_noise', BINS_ALL, False),
}

def result(name, mg):
    make, w, bins, chrom = CONFIGS[name]
    def compute():
        st.binnings = bins
        try:
            return tc.fused(make(mg), w, chromatic=chrom)
        finally:
            st.binnings = BINS_ALL
    return st.cached(f'twocolour {name} mag{mg:g}', compute, extra=(bins,))

def band(name, mg):
    make, _, _, chrom = CONFIGS[name]
    if not chrom:
        return None
    return st.cached(f'twocolour band {name} mag{mg:g}', lambda: tc.band_contrast(result(name, mg), make(mg)))

rows = []
for name in CONFIGS:
    for mg in MAGS:
        r = result(name, mg)
        b = band(name, mg)
        rows.append(tc.row(r, config=name, mag=mg, w_blue_zone=r.params.get('w0_zone', np.nan),
                           contrast_band=b['band'][0] if b else np.nan))
T = pd.DataFrame(rows)
T['band/600'] = T['contrast_band'] / T['contrast']
best = T[T.config.isin(['single 900 rMod 3', 'single 900 unmod'])].groupby('mag')['contrast'].min()
best_band = T[T.config.isin(['single 900 rMod 3', 'single 900 unmod'])].groupby('mag')['contrast_band'].min()
T['vs_best900'] = T['contrast'] / T['mag'].map(best)
T['band_vs_best900'] = T['contrast_band'] / T['mag'].map(best_band)
T.pivot_table(index='mag', columns='config', values='vs_best900', sort=False).round(3)

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(16, 4.2), constrained_layout=True)
for name, d in T.groupby('config', sort=False):
    ls = '--' if 'achromatic' in name else '-'
    axs[0].semilogy(d['mag'], d['contrast'], ls, marker='o', label=name)
    axs[1].plot(d['mag'], d['vs_best900'], ls, marker='o', label=name)
    axs[2].plot(d['mag'], d['chrom_nm'], ls, marker='o', label=name)
axs[0].set_ylabel('raw contrast 2-5 l/D at 600 nm'); axs[1].set_ylabel('contrast / best single 900 nm'); axs[1].axhline(1, color='0.5', lw=1)
axs[2].set_ylabel('chromatic shear [nm rms]')
for ax in axs:
    ax.set_xlabel('magnitude (900 nm)')
axs[1].legend(fontsize=7)
plt.show()
T.pivot_table(index='mag', columns='config', values=['fs', 'chrom_nm', 'total_nm', 'sr'], sort=False).round(3)

In [ ]:
# per-mode weight of the 400-500 nm sensor: min-noise vs per-mode optimum (interpolation limit: dn of the two bands)
w_int = -dn['700-800'] / (dn['400-500'] - dn['700-800'])
fig, axs = plt.subplots(1, 2, figsize=(13, 4), constrained_layout=True, sharey=True)
for ax, mg in zip(axs, (7.0, 11.0)):
    for name in ('fusion min-noise', 'fusion per-mode'):
        w = np.asarray(result(name, mg).weights)[0]
        ax.semilogx(np.arange(1, w.size + 1), w, lw=1, label=name)
    ax.axhline(w_int, color='k', ls=':', lw=1, label=f'shear-free {w_int:.2f}')
    ax.set_title(f'mag {mg:g}', loc='left'); ax.set_xlabel('mode')
axs[0].set_ylabel('weight of the 400-500 nm sensor'); axs[0].legend(fontsize=8)
plt.show()

In [ ]:
# 500-700 nm: the same loop (corrected for 600 nm) evaluated at every science wavelength
LAMS = [k for k in band('fusion per-mode', 7.0) if k != 'band']
fig, axs = plt.subplots(1, 2, figsize=(13, 4), constrained_layout=True)
for ax, mg in zip(axs, (7.0, 11.0)):
    for name in ('single 900 rMod 3', 'single 900 unmod', 'fusion min-noise', 'fusion per-mode'):
        b = band(name, mg)
        ax.semilogy(LAMS, [b[l][0] for l in LAMS], marker='o', label=name)
    ax.set_title(f'mag {mg:g}', loc='left'); ax.set_xlabel('science wavelength [nm]')
axs[0].set_ylabel('raw contrast 2-5 l/D'); axs[0].legend(fontsize=8)
plt.show()
T.pivot_table(index='mag', columns='config', values=['band/600', 'band_vs_best900'], sort=False).round(3)